In [45]:
import weaviate

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=weaviate.auth.AuthApiKey(os.environ["WEAVIATE_API_KEY"]),
    additional_config=weaviate.config.AdditionalConfig(
        timeout=weaviate.config.Timeout(query=6000)
    ),
)

text_weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["TEXT_WEAVIATE_URL"],
    auth_credentials=weaviate.auth.AuthApiKey(os.environ["TEXT_WEAVIATE_API_KEY"])
)

images_collection = weaviate_client.collections.get("IRPapersImages_Default")
text_collection = text_weaviate_client.collections.get("IRPapersText_Default")

/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_68688/719647698.py:17: ResourceWarning: unclosed <ssl.SSLSocket fd=116, family=2, type=1, proto=0, laddr=('10.0.0.233', 56676), raddr=('35.201.124.182', 443)>
  text_collection = text_weaviate_client.collections.get("IRPapersText_Default")


In [46]:
images_collection.config.get().vector_config["default"].vector_index_config

_VectorIndexConfigHNSW(multi_vector=_MultiVectorConfig(encoding=None, aggregation='maxSim'), quantizer=_RQConfig(cache=None, bits=8, rescore_limit=20), cleanup_interval_seconds=300, distance_metric=<VectorDistances.COSINE: 'cosine'>, dynamic_ef_min=100, dynamic_ef_max=500, dynamic_ef_factor=8, ef=-1, ef_construction=128, filter_strategy=<VectorFilterStrategy.ACORN: 'acorn'>, flat_search_cutoff=40000, max_connections=32, skip=False, vector_cache_max_objects=1000000000000)

In [55]:
from pydantic import BaseModel

class ResultWithScore(BaseModel):
    dataset_id: str
    score: float

def hybrid_text_search(query: str) -> list[ResultWithScore]:
    response = text_collection.query.hybrid(
        query=query,
        return_metadata=["score"],
        limit=20,
    )
    return [
        ResultWithScore(
            dataset_id=o.properties["dataset_id"], 
            score=o.metadata.score
        ) 
        for o in response.objects
    ]

def image_search(query: str) -> list[ResultWithScore]:
    response = images_collection.query.near_text(
        query=query,
        return_metadata=["distance"],
        limit=20,
    )
    return [
        ResultWithScore(
            dataset_id=o.properties["dataset_id"], 
            score=-o.metadata.distance  # Convert distance to similarity
        ) 
        for o in response.objects
    ]

def reciprocal_rank_fusion(
    result_lists: list[list[ResultWithScore]], 
    alpha: float = 0.5,
    k: int = 60
) -> list[ResultWithScore]:
    """
    Combine two result lists using Reciprocal Rank Fusion.
    
    Args:
        result_lists: [text_results, image_results]
        alpha: Weight for image search (0 = text only, 1 = image only)
        k: RRF constant (default 60)
    """
    if len(result_lists) != 2:
        raise ValueError("Reciprocal rank fusion expects exactly 2 result lists")
    
    text_results, image_results = result_lists
    scores: dict[str, float] = {}
    
    for rank, result in enumerate(text_results):
        if result.dataset_id not in scores:
            scores[result.dataset_id] = 0
        scores[result.dataset_id] += (1 - alpha) / (k + rank + 1)
    
    for rank, result in enumerate(image_results):
        if result.dataset_id not in scores:
            scores[result.dataset_id] = 0
        scores[result.dataset_id] += alpha / (k + rank + 1)
    
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [ResultWithScore(dataset_id=id_, score=score) for id_, score in sorted_items]

def relative_score_fusion(
    result_lists: list[list[ResultWithScore]], 
    alpha: float = 0.5
) -> list[ResultWithScore]:
    """
    Combine two result lists using Relative Score Fusion.
    
    Args:
        result_lists: [text_results, image_results]
        alpha: Weight for image search (0 = text only, 1 = image only)
    """
    if len(result_lists) != 2:
        raise ValueError("Relative score fusion expects exactly 2 result lists")
    
    text_results, image_results = result_lists
    
    def normalize(results: list[ResultWithScore]) -> dict[str, float]:
        if not results:
            return {}
        
        scores = [r.score for r in results]
        min_score = min(scores)
        max_score = max(scores)
        score_range = max_score - min_score
        
        if score_range == 0:
            return {r.dataset_id: 1.0 for r in results}
        
        return {
            r.dataset_id: (r.score - min_score) / score_range 
            for r in results
        }
    
    normalized_text = normalize(text_results)
    normalized_image = normalize(image_results)
    
    all_ids = set(normalized_text.keys()) | set(normalized_image.keys())
    
    combined_scores: dict[str, float] = {}
    for doc_id in all_ids:
        text_score = normalized_text.get(doc_id, 0.0)
        image_score = normalized_image.get(doc_id, 0.0)
        combined_scores[doc_id] = (1 - alpha) * text_score + alpha * image_score
    
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    return [ResultWithScore(dataset_id=id_, score=score) for id_, score in sorted_items]


# Test it
text_results = hybrid_text_search("What is HyDE?")
print("Text results:")
for r in text_results[:5]:
    print(f"\t{r.dataset_id}: {r.score:.4f}")

image_results = image_search("What is HyDE?")
print("\nImage results:")
for r in image_results[:5]:
    print(f"\t{r.dataset_id}: {r.score:.4f}")

rrf_results = reciprocal_rank_fusion([text_results, image_results])
print("\nReciprocal Rank Fusion:")
for r in rrf_results[:5]:
    print(f"\t{r.dataset_id}: {r.score:.4f}")

rsf_results = relative_score_fusion([text_results, image_results])
print("\nRelative Score Fusion:")
for r in rsf_results[:5]:
    print(f"\t{r.dataset_id}: {r.score:.4f}")

Text results:
	4_5: 0.9148
	4_9: 0.8953
	4_8: 0.8644
	4_2: 0.8247
	4_7: 0.7838

Image results:
	4_9: 13.3626
	4_2: 13.2024
	4_5: 13.1040
	4_4: 12.9182
	4_7: 12.8158

Reciprocal Rank Fusion:
	4_9: 0.0163
	4_5: 0.0161
	4_2: 0.0159
	4_7: 0.0154
	4_8: 0.0153

Relative Score Fusion:
	4_9: 0.9868
	4_5: 0.9406
	4_2: 0.9021
	4_8: 0.7925
	4_7: 0.7856


In [48]:
from datasets import load_dataset
import time

In [57]:
def multimodal_hybrid_search(query: str) -> tuple[list[ResultWithScore], list[ResultWithScore], dict[float, list[ResultWithScore]], dict[float, list[ResultWithScore]]]:
    text_results = hybrid_text_search(query)
    image_results = image_search(query)
    
    alphas = [0.25, 0.5, 0.75]
    rrf_results = {alpha: reciprocal_rank_fusion([text_results, image_results], alpha=alpha) for alpha in alphas}
    rsf_results = {alpha: relative_score_fusion([text_results, image_results], alpha=alpha) for alpha in alphas}
    
    return text_results, image_results, rrf_results, rsf_results

queries = load_dataset("weaviate/irpapers-queries")["train"]

def compute_success_at_k(results: list[ResultWithScore], gold_id: str, k: int) -> bool:
    return gold_id in [r.dataset_id for r in results[:k]]

alphas = [0.25, 0.5, 0.75]

# Track successes for each method
text_successes = {1: 0, 5: 0, 20: 0}
image_successes = {1: 0, 5: 0, 20: 0}
rrf_successes = {alpha: {1: 0, 5: 0, 20: 0} for alpha in alphas}
rsf_successes = {alpha: {1: 0, 5: 0, 20: 0} for alpha in alphas}

# Track failed query indices (only text and image for analysis)
text_failures = {1: set(), 5: set(), 20: set()}
image_failures = {1: set(), 5: set(), 20: set()}

query_times = []

experiment_start = time.time()
for idx, query in enumerate(queries):
    if idx % 20 == 19:
        print(f"Processing query {idx + 1} of {len(queries)}")
        print(f"  Text  - S@1: {text_successes[1] / (idx + 1):.3f}, S@5: {text_successes[5] / (idx + 1):.3f}, S@20: {text_successes[20] / (idx + 1):.3f}")
        print(f"  Image - S@1: {image_successes[1] / (idx + 1):.3f}, S@5: {image_successes[5] / (idx + 1):.3f}, S@20: {image_successes[20] / (idx + 1):.3f}")
        for alpha in alphas:
            print(f"  RRF(α={alpha}) - S@1: {rrf_successes[alpha][1] / (idx + 1):.3f}, S@5: {rrf_successes[alpha][5] / (idx + 1):.3f}, S@20: {rrf_successes[alpha][20] / (idx + 1):.3f}")
            print(f"  RSF(α={alpha}) - S@1: {rsf_successes[alpha][1] / (idx + 1):.3f}, S@5: {rsf_successes[alpha][5] / (idx + 1):.3f}, S@20: {rsf_successes[alpha][20] / (idx + 1):.3f}")
        print(f"Experiment running for {time.time() - experiment_start:.2f} seconds")
        print(f"Average query time: {sum(query_times) / len(query_times):.2f} seconds")

    start_time = time.time()
    text_results, image_results, rrf_results, rsf_results = multimodal_hybrid_search(query["question"])
    query_time = time.time() - start_time
    query_times.append(query_time)

    gold_id = query["dataset_id"]
    for k in [1, 5, 20]:
        if compute_success_at_k(text_results, gold_id, k):
            text_successes[k] += 1
        else:
            text_failures[k].add(idx)
        
        if compute_success_at_k(image_results, gold_id, k):
            image_successes[k] += 1
        else:
            image_failures[k].add(idx)
        
        for alpha in alphas:
            if compute_success_at_k(rrf_results[alpha], gold_id, k):
                rrf_successes[alpha][k] += 1
            
            if compute_success_at_k(rsf_results[alpha], gold_id, k):
                rsf_successes[alpha][k] += 1

print("\n" + "="*60)
print("Final Results")
print("="*60)
print(f"  Text  - S@1: {text_successes[1] / len(queries):.3f}, S@5: {text_successes[5] / len(queries):.3f}, S@20: {text_successes[20] / len(queries):.3f}")
print(f"  Image - S@1: {image_successes[1] / len(queries):.3f}, S@5: {image_successes[5] / len(queries):.3f}, S@20: {image_successes[20] / len(queries):.3f}")
for alpha in alphas:
    print(f"  RRF(α={alpha}) - S@1: {rrf_successes[alpha][1] / len(queries):.3f}, S@5: {rrf_successes[alpha][5] / len(queries):.3f}, S@20: {rrf_successes[alpha][20] / len(queries):.3f}")
    print(f"  RSF(α={alpha}) - S@1: {rsf_successes[alpha][1] / len(queries):.3f}, S@5: {rsf_successes[alpha][5] / len(queries):.3f}, S@20: {rsf_successes[alpha][20] / len(queries):.3f}")

# Multimodal strengths and weaknesses analysis
print("\n" + "="*60)
print("Multimodal Analysis: Text vs Image")
print("="*60)

for k in [1, 5, 20]:
    only_text_fails = text_failures[k] - image_failures[k]  # Text fails, Image succeeds
    only_image_fails = image_failures[k] - text_failures[k]  # Image fails, Text succeeds
    both_fail = text_failures[k] & image_failures[k]
    both_succeed = set(range(len(queries))) - text_failures[k] - image_failures[k]
    
    print(f"\n--- Success@{k} ---")
    print(f"  Image saves text (only text fails): {len(only_text_fails)} queries")
    print(f"    {sorted(only_text_fails)[:10]}{'...' if len(only_text_fails) > 10 else ''}")
    print(f"  Text saves image (only image fails): {len(only_image_fails)} queries")
    print(f"    {sorted(only_image_fails)[:10]}{'...' if len(only_image_fails) > 10 else ''}")
    print(f"  Both fail: {len(both_fail)} queries")
    print(f"  Both succeed: {len(both_succeed)} queries")

Processing query 20 of 180
  Text  - S@1: 0.400, S@5: 0.650, S@20: 0.800
  Image - S@1: 0.500, S@5: 0.700, S@20: 0.800
  RRF(α=0.25) - S@1: 0.450, S@5: 0.650, S@20: 0.800
  RSF(α=0.25) - S@1: 0.500, S@5: 0.650, S@20: 0.850
  RRF(α=0.5) - S@1: 0.450, S@5: 0.700, S@20: 0.850
  RSF(α=0.5) - S@1: 0.550, S@5: 0.650, S@20: 0.850
  RRF(α=0.75) - S@1: 0.550, S@5: 0.750, S@20: 0.800
  RSF(α=0.75) - S@1: 0.600, S@5: 0.700, S@20: 0.850
Experiment running for 79.22 seconds
Average query time: 4.17 seconds
Processing query 40 of 180
  Text  - S@1: 0.325, S@5: 0.625, S@20: 0.850
  Image - S@1: 0.400, S@5: 0.700, S@20: 0.825
  RRF(α=0.25) - S@1: 0.350, S@5: 0.650, S@20: 0.850
  RSF(α=0.25) - S@1: 0.400, S@5: 0.650, S@20: 0.850
  RRF(α=0.5) - S@1: 0.325, S@5: 0.700, S@20: 0.875
  RSF(α=0.5) - S@1: 0.450, S@5: 0.650, S@20: 0.850
  RRF(α=0.75) - S@1: 0.400, S@5: 0.750, S@20: 0.825
  RSF(α=0.75) - S@1: 0.450, S@5: 0.675, S@20: 0.850
Experiment running for 154.40 seconds
Average query time: 3.96 seconds
P